# 02. LSTM 감성분류기 학습 (비교군)

직접 학습하는 LSTM 베이스라인. DistilBERT와 같은 데이터·지표(Accuracy/F1)로 비교한다.

산출물: `models/lstm/{model.pt, vocab.json, metrics.json}`

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from src.config import DATA_DIR, MODEL_DIR, MAX_LEN, RANDOM_SEED
from src.models.dataset import build_vocab, encode
from src.models.lstm import LSTMClassifier

EPOCHS = 3
torch.manual_seed(RANDOM_SEED)

In [2]:
tr = pd.read_csv(DATA_DIR / "train.csv")
te = pd.read_csv(DATA_DIR / "test.csv")
vocab = build_vocab(tr["text"].tolist())
print(f"train={len(tr)} test={len(te)} vocab={len(vocab):,}")

train=11657 test=2498 vocab=21,114


In [3]:
def tensors(df, vocab):
    X = torch.tensor([encode(t, vocab, MAX_LEN) for t in df["text"]])
    y = torch.tensor(df["label"].tolist())
    return TensorDataset(X, y)

dl = DataLoader(tensors(tr, vocab), batch_size=64, shuffle=True)
model = LSTMClassifier(len(vocab))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

for ep in range(EPOCHS):
    model.train()
    total = 0.0
    for X, y in dl:
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
        total += loss.item() * len(y)
    print(f"epoch {ep + 1}/{EPOCHS} — avg loss {total / len(tr):.4f}")

epoch 1/3 — avg loss 0.4864


epoch 2/3 — avg loss 0.4651


epoch 3/3 — avg loss 0.4595


In [4]:
model.eval()
Xte = torch.tensor([encode(t, vocab, MAX_LEN) for t in te["text"]])
with torch.no_grad():
    preds = model(Xte).argmax(1).tolist()

result = {"accuracy": accuracy_score(te["label"], preds),
          "f1": f1_score(te["label"], preds)}
print("LSTM test:", result)
print("confusion matrix:\n", confusion_matrix(te["label"], preds))

LSTM test: {'accuracy': 0.8178542834267414, 'f1': 0.8995363214837713}
confusion matrix:
 [[   6  441]
 [  14 2037]]


In [5]:
out = MODEL_DIR / "lstm"
out.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out / "model.pt")
(out / "vocab.json").write_text(json.dumps(vocab))
(out / "metrics.json").write_text(json.dumps(result, indent=2))
print(f"saved → {out}")

saved → /Users/gomuseo/Desktop/Python/review-check/models/lstm
